# Replay Data Analysis

In [2]:
from collections import Counter
import logging
import glob

import pandas as pd
import sc2reader
from sc2reader.events.tracker import UnitBornEvent, UnitDiedEvent, UnitInitEvent, UnitDoneEvent, UpgradeCompleteEvent, UnitTypeChangeEvent

In [3]:
logger = logging.getLogger(__name__)


TRACKED_UNIT_TYPES = {
    "Protoss": [
        "Probe",
        "Zealot",
        "Stalker",
        "Sentry",
        "Adept",
        "HighTemplar",
        "DarkTemplar",
        "Immortal",
        "Colossus",
        "Disruptor",
        "Archon",
        "Observer",
        "WarpPrism",
        "Phoenix",
        "VoidRay",
        "Oracle",
        "Carrier",
        "Tempest",
        "MothershipCore",
    ],
    "Terran": [
        "SCV",
        "MULE",
        "Marine",
        "Marauder",
        "Reaper",
        "Ghost",
        "Hellion",
        "Hellbat",
        "SiegeTank",
        "Cyclone",
        "WidowMine",
        "Thor",
        "Viking",
        "Medivac",
        "Liberator",
        "Ravem",
        "Banshee",
        "Battlecruiser",
    ],
    "Zerg": [
        "Drone",
        "Queen",
        "Zergling",
        "Baneling",
        "Roach",
        "Ravager",
        "Hydralisk",
        "Lurker",
        "LurkerMP",
        "Infestor",
        "SwarmHost",
        "Ultralish",
        "Overlord",
        "Overseer",
        "Mutalisk",
        "Corruptor",
        "BroodLord",
        "Viper",
    ],
}

TRACKED_UPGRADE_TYPES = {
    "Protoss": [
        "WarpGateResearch",
        "BlinkTech",
        "Charge",
        "ProtossGroundWeaponsLevel1",
        "ProtossGroundWeaponsLevel2",
        "ProtossGroundWeaponsLevel3",
        "PsiStormTech",
        "ExtendedThermalLance",
        "ProtossGroundArmorsLevel1",
        "ProtossGroundArmorsLevel2",
        "ProtossGroundArmorsLevel3",
        "ProtossShieldsLevel1",
        "ProtossShieldsLevel2",
        "ProtossShieldsLevel3",
        "GraviticDrive",
        "DarkTemplarBlinkUpgrade",
        "ObserverGraviticBooster",
        "ProtossAirWeaponsLevel1",
        "ProtossAirWeaponsLevel2",
        "ProtossAirWeaponsLevel3",
        "AdeptPiercingAttack",
        "TempestGroundAttackUpgrade",
    ],
    "Terran": [
        "Stimpack",
        "ShieldWall",
        "PunisherGrenades",
        "InterferenceMatrix"
        "TerranInfantryWeaponsLevel1",
        "TerranInfantryWeaponsLevel2",
        "TerranInfantryWeaponsLevel3",
        "TerranInfantryArmorsLevel1",
        "TerranInfantryArmorsLevel2",
        "TerranInfantryArmorsLevel3",
        "TerranVehicleWeaponsLevel1",
        "TerranVehicleWeaponsLevel2",
        "TerranVehicleWeaponsLevel3",
        "TerranShipWeaponsLevel1",
        "TerranShipWeaponsLevel2",
        "TerranShipWeaponsLevel3",
        "TerranVehicleAndShipArmorsLevel1",
        "TerranVehicleAndShipArmorsLevel2",
        "TerranVehicleAndShipArmorsLevel3",
        "BansheeCloak",
        "CycloneLockOnDamageUpgrade",
        "PersonalCloaking",
        "HighCapacityBarrels",
        "TerranBuildingArmor",
        "LiberatorAGRangeUpgrade",
        "SmartServos",
        "DrillClaws",
        "HiSecAutoTracking",
        "MedivacCaduceusReactor",
        "BansheeSpeed",
        "BattlecruiserEnableSpecializations",
    ],
    "Zerg": [
        "zerglingmovementspeed",
        "ZergMeleeWeaponsLevel1",
        "ZergMeleeWeaponsLevel2",
        "ZergMeleeWeaponsLevel3",
        "GlialReconstitution",
        "overlordspeed",
        "ZergGroundArmorsLevel1",
        "ZergGroundArmorsLevel2",
        "ZergGroundArmorsLevel3",
        "ZergMissileWeaponsLevel1",
        "ZergMissileWeaponsLevel2",
        "ZergMissileWeaponsLevel3",
        "CentrificalHooks",
        "EvolveGroovedSpines",
        "zerglingattackspeed",
        "EvolveMuscularAugments",
        "Burrow",
        "LurkerRange",
        "DiggingClaws",
        "ChitinousPlating",
        "Frenzy",
        "ZergFlyerWeaponsLevel1",
        "ZergFlyerWeaponsLevel2",
        "ZergFlyerWeaponsLevel3",
        "AnabolicSynthesis",
        "NeuralParasite",
        "ZergFlyerArmorsLevel1",
        "ZergFlyerArmorsLevel2",
        "ZergFlyerArmorsLevel3",
        "TunnelingClaws",
    ],
}

In [4]:
class UnitTracker:
    """Class for tracking units in a StarCraft II replay.

    This class handles all unit-related events and maintains the current state of units for both players.
    It tracks units that are born, die, change type, or are initialized and completed.

    Attributes
    ----------
    players : List[sc2reader.objects.Participant]
        List of players in the replay.
    tracked_units : Dict[int, Dict[int, str]]
        Dictionary containing the currently tracked units for both players, indexed by player ID.
        Format: {player_id: {unit_id: unit_type_name, ...}}
    tracked_initialisations : Dict[int, Dict[int, str]]
        Dictionary containing the currently tracked initialisations for both players, indexed by player ID.
        Format: {player_id: {unit_id: unit_type_name, ...}}
    tracked_upgrades : Dict[int, Dict[int, str]]
        Dictionary containing the currently tracked upgrades for both players, indexed by player ID.
        Format: {player_id: {upgrade_id: upgrade_type_name, ...}}

    """

    def __init__(self, players: list[sc2reader.objects.Participant]) -> None:
        """Initialize the UnitTracker.

        Parameters
        ----------
        players : List[sc2reader.objects.Participant]
            List of players in the replay.

        """
        self.players = players
        self.tracked_units = {0: {}, 1: {}}
        self.tracked_initialisations = {0: {}, 1: {}}
        self.tracked_upgrades = {0: {}, 1: {}}

    def handle_unit_born(self, event: UnitBornEvent) -> None:
        """Handle a UnitBornEvent and update the units dictionary.

        Parameters
        ----------
        event : UnitBornEvent
            The UnitBornEvent to process.

        """
        if event.control_pid == 0:  # Some events during the game setup are not assigned to player 1 or player 2
            return

        player = self.players[event.control_pid - 1]

        if event.unit_type_name in TRACKED_UNIT_TYPES[player.play_race]:
            self.tracked_units[event.control_pid - 1][
                event.unit_id
            ] = event.unit_type_name

    def handle_unit_died(self, event: UnitDiedEvent) -> None:
        """Handle a UnitDiedEvent and update the units dictionary.

        UnitDiedEvents do not have a player id, so we attempt to remove the unit from both players' tracked units.

        Parameters
        ----------
        event : UnitDiedEvent
            The UnitDiedEvent to process.

        """
        try:
            self.tracked_units[0].pop(event.unit_id)
        except KeyError:
            try:
                self.tracked_units[1].pop(event.unit_id)
            except KeyError:
                logger.warning("Unit not found in tracked units for both players.", extra={"unit_id": event.unit_id})

    def handle_unit_type_change(self, event: UnitTypeChangeEvent) -> None:
        """Handle a UnitTypeChangeEvent and update the units dictionary.

        UnitTypeChangeEvents do not have a player id, so we attempt to update the unit type for both players' tracked
        units. UnitTypeChangeEvents modify the unit_type_name for an already existing unit_id.

        Parameters
        ----------
        event : UnitTypeChangeEvent
            The UnitTypeChangeEvent to process.

        """
        not_found_unit = 0

        for i, player in enumerate(self.players):
            if event.unit_type_name in TRACKED_UNIT_TYPES[player.play_race]:
                try:
                    self.tracked_units[i][event.unit_id] = event.unit_type_name
                except KeyError:
                    not_found_unit += 1

        if not_found_unit > 1:
            logger.warning(
                "Warning: UnitTypeChangeEvent unit not found in tracked units for both players.",
                extra={"unit_id": event.unit_id},
            )

In [5]:
replay_path = "/Users/nedwebster/Documents/python_projects/personal_projects/starcraft_predictor/data/replays/**/*.SC2Replay"
replay_paths = glob.glob(replay_path, recursive=True)

In [6]:
for i, replay_path in enumerate(replay_paths):
    print(i, end="\r")
    replay = sc2reader.load_replay(replay_path)
    races = set([x.play_race for x in replay.players])
    if races == set(["Protoss", "Zerg"]):
        unit_born_events = [x for x in replay.events if isinstance(x, UnitBornEvent)]
        units = [x for x in unit_born_events if "mothership" in x.unit.name.lower()]
        if len(units) > 0:
            print(replay_path)
            print(units[0].unit_type_name, units[0].unit.name)

/Users/nedwebster/Documents/python_projects/personal_projects/starcraft_predictor/data/replays/2024_03-ESL_SC2_Masters_Spring_2024_Finals/3 - Knockout Bracket/6 - Astrea vs Scarlett/20240601 - Game 1 - Astrea vs Scarlett - PvZ - Goldenaura.SC2Replay
Mothership Mothership
/Users/nedwebster/Documents/python_projects/personal_projects/starcraft_predictor/data/replays/2024_03-ESL_SC2_Masters_Spring_2024_Finals/3 - Knockout Bracket/6 - Astrea vs Scarlett/20240601 - Game 3 - Astrea vs Scarlett - PvZ - Site Delta.SC2Replay
Mothership Mothership


KeyboardInterrupt: 

In [7]:
for i, replay_path in enumerate(replay_paths):
    print(i, end="\r")
    replay = sc2reader.load_replay(replay_path)
    races = set([x.play_race for x in replay.players])
    if races == set(["Protoss", "Zerg"]):
        unit_type_change_events = [x for x in replay.events if isinstance(x, UnitTypeChangeEvent)]
        units = [x for x in unit_type_change_events if "mothership" in x.unit_type_name.lower()]
        if len(units) > 0:
            print(replay_path)
            print(units[0].unit_type_name, units[0].unit.name)
            

KeyboardInterrupt: 

In [8]:
for i, replay_path in enumerate(replay_paths):
    print(i, end="\r")
    replay = sc2reader.load_replay(replay_path)
    races = set([x.play_race for x in replay.players])
    if races == set(["Protoss", "Zerg"]):
        unit_done_events = [x for x in replay.events if isinstance(x, UnitDoneEvent)]
        units = [x for x in unit_done_events if "mothership" in x.unit.name.lower()]
        if len(units) > 0:
            print(replay_path)
            print(units[0].unit_type_name, units[0].unit.name)

KeyboardInterrupt: 

In [11]:
replay_path = "/Users/nedwebster/Documents/python_projects/personal_projects/starcraft_predictor/data/replays/2024_03-ESL_SC2_Masters_Spring_2024_Finals/4 - Playoffs/2 - ByuN vs Dark/20240601 - Game 3 - Dark vs ByuN - ZvT - Site Delta.SC2Replay"

In [12]:
replay = sc2reader.load_replay(replay_path)

In [13]:
unit_born_events = [x for x in replay.events if isinstance(x, UnitBornEvent)]
unit_type_change_events = [x for x in replay.events if isinstance(x, UnitTypeChangeEvent)]
unit_died_events = [x for x in replay.events if isinstance(x, UnitDiedEvent)]
unit_init_events = [x for x in replay.events if isinstance(x, UnitInitEvent)]
unit_done_events = [x for x in replay.events if isinstance(x, UnitDoneEvent)]


In [14]:
Counter([x.unit_type_name for x in unit_born_events])

Counter({'InvisibleTargetDummy': 3959,
         'Zergling': 814,
         'Larva': 607,
         'Marine': 258,
         'Drone': 124,
         'SCV': 80,
         'MineralField': 56,
         'MineralField750': 56,
         'Hydralisk': 49,
         'VespeneGeyser': 28,
         'Broodling': 27,
         'Marauder': 24,
         'Ghost': 24,
         'Overlord': 23,
         'Medivac': 22,
         'WidowMine': 20,
         'Queen': 18,
         'Changeling': 15,
         'MULE': 10,
         'ChangelingMarineShield': 10,
         'LabMineralField750': 8,
         'LabMineralField': 8,
         'Hellion': 6,
         'SpacePlatformGeyser': 4,
         'DestructibleRock6x6': 4,
         'Reaper': 4,
         'SiegeTank': 4,
         'UnbuildableRocksDestructible': 2,
         'DestructibleRampDiagonalHugeBLUR': 2,
         'BeaconArmy': 2,
         'BeaconDefend': 2,
         'BeaconAttack': 2,
         'BeaconHarass': 2,
         'BeaconIdle': 2,
         'BeaconAuto': 2,
         'Be

In [15]:
Counter([x.unit_type_name for x in unit_type_change_events])

Counter({'Egg': 598,
         'Larva': 592,
         'LurkerMP': 98,
         'LurkerMPBurrowed': 92,
         'BanelingCocoon': 91,
         'CreepTumorBurrowed': 86,
         'Baneling': 82,
         'LurkerMPEgg': 28,
         'WidowMineBurrowed': 19,
         'SiegeTankSieged': 16,
         'SiegeTank': 13,
         'OrbitalCommand': 9,
         'SupplyDepotLowered': 9,
         'HellionTank': 6,
         'CommandCenterFlying': 6,
         'Reactor': 5,
         'OrbitalCommandFlying': 5,
         'Overseer': 5,
         'CommandCenter': 5,
         'SporeCrawlerUprooted': 4,
         'SporeCrawler': 4,
         'LiberatorAG': 4,
         'FactoryFlying': 3,
         'Factory': 3,
         'SupplyDepot': 3,
         'OverlordCocoon': 3,
         'OverseerSiegeMode': 3,
         'FactoryReactor': 2,
         'StarportFlying': 2,
         'Starport': 2,
         'PlanetaryFortress': 2,
         'WidowMine': 2,
         'Liberator': 2,
         'TechLab': 2,
         'BarracksFlying':

In [17]:
lurker = [x for x in unit_type_change_events if "lurker" in x.unit_type_name.lower()][0]
print(lurker.unit_id)

150732803


In [21]:
unit_born_event = [x for x in unit_born_events if x.unit.id == lurker.unit_id]
print(len(unit_born_event))
unit_born_event[0].unit_type_name, unit_born_event[0].unit.name

1


('Hydralisk', 'LurkerBurrowed')

In [26]:
unit_type_change_events = [x for x in unit_type_change_events if x.unit.id == lurker.unit_id]
print(len(unit_type_change_events))
[(x.unit_type_name, x.second) for x in unit_type_change_events]

17


[('LurkerMPEgg', 1045),
 ('LurkerMP', 1070),
 ('LurkerMPBurrowed', 1080),
 ('LurkerMP', 1102),
 ('LurkerMPBurrowed', 1142),
 ('LurkerMP', 1198),
 ('LurkerMPBurrowed', 1202),
 ('LurkerMP', 1232),
 ('LurkerMPBurrowed', 1251),
 ('LurkerMP', 1262),
 ('LurkerMPBurrowed', 1270),
 ('LurkerMP', 1294),
 ('LurkerMPBurrowed', 1296),
 ('LurkerMP', 1337),
 ('LurkerMPBurrowed', 1369),
 ('LurkerMP', 1411),
 ('LurkerMPBurrowed', 1413)]